# Banking Intent Assistant on BANKING77

PCAM ZG516 Applied Generative AI, Group Major Project, Assignment 1.

This guided notebook covers the dataset, Qwen causal language model, prompting and decoding experiments, structured output, retrieval (RAG), the separate LoRA/PEFT option, the causal-LM/VAE/GAN family comparison, error analysis, and the final banking assistant.

Each code cell is preceded by a short explanation. The two evaluation subsets each contain 100 distinct examples and include all 77 intents. The development subset is for experiments; the held-out subset is reserved for final evaluation.

The full Qwen-based notebook requires a machine with sufficient memory; a Colab GPU runtime is recommended. PEFT training is optional and is not launched during Run All unless explicitly enabled.

## 0. Setup

In Google Colab, the setup cell below clones the project from the requested GitHub repository, installs its runtime dependencies, and optionally links the experiment-results folder to Google Drive. Locally, it uses the checked-out project folder. Select a GPU runtime in Colab before loading Qwen or running LoRA training.

Configure the Colab/local environment, clone this exact repository in Colab, install runtime packages, and prepare the results folder.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Set this before importing src.llm if you want to use another compatible chat model.
# os.environ["MODEL"] = "Qwen/Qwen2.5-3B-Instruct"

if "google.colab" in sys.modules:
    repo_url = "https://github.com/mohit-tendolkar-bits/Banking_Agent_Group23_BITS.git"
    repo_dir = Path("/content/Banking_Agent_Group23_BITS")
    if not (repo_dir / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
    os.chdir(repo_dir)
    %pip install -q transformers torch peft jsonschema pandas scikit-learn

    from google.colab import drive
    drive.mount("/content/drive")
    drive_results = Path("/content/drive/MyDrive/banking-agent-results") / os.environ.get("MODEL", "Qwen/Qwen2.5-1.5B-Instruct").split("/")[-1]
    drive_results.mkdir(parents=True, exist_ok=True)
    local_results = Path("evaluation/results")
    if local_results.is_symlink():
        local_results.unlink()
    elif local_results.exists():
        shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
        shutil.rmtree(local_results)
    local_results.symlink_to(drive_results, target_is_directory=True)

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
pd.set_option("display.max_colwidth", 80)


## 1. Dataset and evaluation splits (Req 2)

BANKING77 contains 13,083 customer messages across 77 intents, with fixed training and test files. The training split supplies examples for training and retrieval. To make the evaluation cover every label, the generator reserves one different test example per intent for each subset, then fills each 100-row subset with 23 additional unused test examples.

The development and held-out files are reproducible and disjoint. Experiments use only the development file; the final system is evaluated once on the held-out file.

Create deterministic development and held-out samples with all 77 intents represented in each 100-row split.


In [ ]:
%run -m evaluation.make_eval_subset


Load the official dataset and both evaluation samples, then verify their sizes, label coverage, and separation.


In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
subset = pd.read_csv("evaluation/eval_subset.csv")
heldout_subset = pd.read_csv("evaluation/heldout_subset.csv")

print(f"train: {len(train)} rows, {train.category.nunique()} intents")
print(f"official test: {len(test)} rows, {test.category.nunique()} intents")
print(f"development: {len(subset)} rows, {subset.category.nunique()} intents")
print(f"held-out: {len(heldout_subset)} rows, {heldout_subset.category.nunique()} intents")
print(f"overlap between development and held-out: {len(set(subset.text) & set(heldout_subset.text))}")
print(f"message length (words): mean {train.text.str.split().str.len().mean():.1f}, max {train.text.str.split().str.len().max()}")
subset.head(8)


Inspect the training-label distribution and check that all 77 intent names are present.


In [ ]:
# Show the most and least frequent training labels and verify the label set size.
counts = train.category.value_counts()
display(pd.concat([counts.head(5), counts.tail(5)]).to_frame("training examples"))
print(f"unique intent labels: {train.category.nunique()}")
print("labels with a capital letter:", [label for label in counts.index if label != label.lower()])


## 2. Causal language model (Req 4, Req 5)

`src/llm.py` loads `Qwen/Qwen2.5-1.5B-Instruct`, a pretrained causal (autoregressive) language model. The same `generate()` helper is used by the experiments and assistant. The next cell illustrates tokenization, embeddings, next-token logits, and text generation. Loading Qwen requires substantial memory; use a GPU runtime if CPU memory is limited.

Load Qwen and demonstrate how a customer message is tokenized and converted into next-token predictions.


In [ ]:
import torch
from src.llm import tokenizer, model, generate, DEVICE, MODEL

message = "How do I track my card?"
ids = tokenizer(message)["input_ids"]
print("1. tokenization:", [tokenizer.decode([i]) for i in ids], "->", ids)

emb = model.get_input_embeddings()(torch.tensor([ids]).to(DEVICE))
print("2. embeddings:  ", tuple(emb.shape), "(1 sequence, tokens, hidden size)")

with torch.no_grad():
    logits = model(torch.tensor([ids]).to(DEVICE)).logits[0, -1]
top = torch.topk(logits.float().softmax(-1), 5)
print("3. LM head, top-5 next tokens after the message:", [(tokenizer.decode([i]), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

print("4. autoregressive generation (greedy, chat format):")
print("  ", generate([{"role": "user", "content": message}], max_new_tokens=40))

## 3. E1 - baseline intent classification (Req 5)

The baseline asks the model to choose from the 77 intent names using a simple prompt and greedy decoding. Since the response is free text, `parse_intent` extracts the longest known label. The experiment evaluates every row in the development set.

Run the baseline experiment. It makes one Qwen generation per development example and saves predictions and correctness flags to `evaluation/results/results_baseline.csv`.

Run the baseline prompt experiment and save one prediction per development example.


In [ ]:
%run -m src.baseline


The next cell loads the baseline output and summarizes accuracy, missing/invalid labels, and example predictions. These scores are on the development subset, not the untouched held-out set.

Inspect the baseline prompt and label parser with a small example; this cell does not run another benchmark.


In [ ]:
from src.baseline import build_prompt, parse_intent, INTENTS

example_prompt = build_prompt("How do I track my card?")
print("number of candidate intents:", len(INTENTS))
print("prompt preview:", example_prompt[0]["content"][-250:])
print("example parsed label:", parse_intent("The best intent is cash_withdrawal"))


## 4. E2 - prompt comparison (Req 6, Req 11)

The prompt experiment compares five versions of the same task: a plain instruction, a role description, a strict output constraint, five labelled examples, and examples plus the constraint. All variants use the same examples and greedy decoding setup for a fair comparison on the development set.

Run the five prompt variants. Results are saved separately as `results_<style>.csv` so the following cell can compare accuracy and output-format behavior.

Run all five prompt-style experiments on the same development messages.


In [ ]:
%run -m src.prompts


Review the prompt-comparison table and sample predictions. Check both whether the intent is correct and whether the model follows the requested answer format.

Preview each prompt template to see how roles, constraints, and examples change the model input.


In [ ]:
from src.prompts import STYLES, build_prompt as prompt_by_style

print("prompt styles:", STYLES)
for style in STYLES:
    prompt = prompt_by_style("How do I track my card?", style)
    print(f"\n{style}: {len(prompt)} message(s)")
    print(prompt[-1]["content"][-180:])


## 5. E3 - decoding comparison (Req 7)

Using the same constrained few-shot prompt, this experiment compares greedy decoding with temperature, top-k, and top-p sampling. It measures how decoding changes label accuracy and output consistency; greedy decoding is used for intent decisions in the final assistant.

Run the decoding experiment. It resets the random seed for sampled settings and saves predictions for each decoding configuration.

Run the decoding-setting experiment and save outputs for greedy and sampling configurations.


In [ ]:
%run -m src.decoding


Inspect decoding accuracy and the number of predictions that changed relative to greedy decoding. Sampling can add response variety but may make a fixed-label prediction less stable.

Display the decoding options so the comparison settings are explicit.


In [ ]:
from src.decoding import SETTINGS

pd.DataFrame([{"setting": name, **options} for name, options in SETTINGS.items()])


## 6. E4 - structured output (Req 9)

The model is asked for a JSON object containing an intent and confidence. A JSON Schema restricts the intent to the 77 known labels. The experiment distinguishes syntactically valid JSON from schema-valid output and correct classification.

Run the structured-output experiment on the development subset. The output CSV records generated text, parsed fields, schema errors, and correctness.

Run the JSON/schema-constrained output experiment on the development split.


In [ ]:
%run -m src.structured


The next cell displays the schema and separates JSON parsing, schema validation, and intent accuracy. A valid JSON response is not necessarily a correct prediction.

Inspect the schema and summarize JSON validity, schema compliance, confidence values, and errors.


In [ ]:
import json
from src.structured import SCHEMA, parse_json

print(json.dumps({**SCHEMA, "properties": {**SCHEMA["properties"], "intent": {"type": "string", "enum": "<77 intent names>"}}}, indent=2))

r = pd.read_csv("evaluation/results/results_structured.csv")
valid = r.error.isna()
json_ok = (~r.error.str.startswith("not JSON", na=False)).sum()
print(f"\nvalid JSON {json_ok}/{len(r)}, pass schema {valid.sum()}/{len(r)}, accuracy all {r.correct.mean():.2f}, accuracy valid rows {r[valid].correct.mean():.2f}")
print("confidence values:", r.confidence.value_counts().to_dict())
print("\nrejected by schema:")
display(r[~valid][["text", "category", "output", "error"]].head(5))
print("validated samples:")
r[valid][["text", "category", "output", "correct"]].head(3)


## 7. Adaptation strategy: prompting vs. RAG vs. PEFT (Req 8)

These are three adaptation strategies for the same Qwen base model, not three different model families:

- **Prompting:** change the instructions or include labelled examples; do not retrieve at inference or update model weights.
- **RAG:** retrieve five similar labelled training examples with TF-IDF and add them to the prompt; do not update model weights.
- **PEFT (LoRA):** train a small adapter on the training split; do not retrieve examples at inference.

Prompting and RAG run below on the same 100-row development set. The PEFT training script is included and can be enabled in the optional cell; its full results are only compared when a completed metrics file exists. Do not treat an unrun or smoke-test PEFT score as a measured result.

The TF-IDF + logistic-regression classifier is a non-generative reference trained on the training split. It is included to provide a classical baseline on the same development examples; it is not one of the three adaptation strategies.

Train and evaluate the TF-IDF/logistic-regression reference classifier using the training and development data.


In [ ]:
%run -m evaluation.reference_classifier


Run the RAG-style retrieval experiment. For each development message, it retrieves five similar examples from training data and evaluates both the LLM using those examples and a 5-nearest-neighbour vote.

Run retrieval-augmented intent classification using similar labelled training examples.


In [ ]:
%run -m src.retrieval


Review the retrieved neighbours and compare the prompt-only, 5-NN, classical, and RAG outputs. Retrieval uses training examples only; evaluation messages are not added to the retrieval index.

Review retrieval examples and compare their outcomes with the retrieved-example vote and other references.


In [ ]:
from src.retrieval import retrieve

r = pd.read_csv("evaluation/results/results_retrieval.csv")
print(f"LLM + retrieved examples: {r.correct.mean():.2f}    5-NN vote only: {r.knn_correct.mean():.2f}    invalid predictions: {r.predicted.isna().sum()}")
print(f"LLM right where vote wrong: {(r.correct & ~r.knn_correct).sum()}, LLM wrong where vote right: {(~r.correct & r.knn_correct).sum()}")

print("\nRetrieved training examples for a sample message:")
display(retrieve("How do I track my card?")[["text", "category"]])


### Compare prompting, RAG, and PEFT on the development split

Prompting and RAG metrics below use the same Qwen base model and the same development examples. PEFT metrics are added only if the standalone LoRA training script has completed and produced its CSV; the held-out PEFT row is reported separately and is not used for tuning.


This cell builds an apples-to-apples development-set table from the saved prompting and RAG predictions, and includes PEFT only when measured metrics are available.


In [ ]:
prompt_metrics = pd.read_csv("evaluation/results/results_few_shot_constraint.csv")
rag_metrics = pd.read_csv("evaluation/results/results_retrieval.csv")
adaptation_rows = [
    {"strategy": "Prompting (few-shot + constraint)", "split": "development", "examples": len(prompt_metrics), "accuracy": prompt_metrics.correct.mean()},
    {"strategy": "RAG (five retrieved examples)", "split": "development", "examples": len(rag_metrics), "accuracy": rag_metrics.correct.mean()},
]
peft_metrics_file = Path("evaluation/results/results_peft_lora.csv")
if peft_metrics_file.exists():
    peft_metrics = pd.read_csv(peft_metrics_file)
    adaptation_rows.extend(
        {"strategy": row.method, "split": row.split, "examples": row.examples, "accuracy": row.accuracy}
        for row in peft_metrics.itertuples()
    )
else:
    print("PEFT accuracy is not reported: the full LoRA training/evaluation run has not produced a metrics file.")

display(pd.DataFrame(adaptation_rows))


### Optional: train the LoRA/PEFT adapter

The implementation in `evaluation/peft_lora_comparison.py` fine-tunes Qwen with LoRA on the training split, then evaluates both development and held-out subsets. It is disabled during Run All because the 1.5B base model requires substantial memory. Use a suitable CUDA GPU; a two-example smoke test is for pipeline checking only and must not be reported as an assignment result.

To train from a terminal instead, run `uv run -m evaluation.peft_lora_comparison` from the repository root.


Set the flag to `True` only when you intend to run full LoRA training on a GPU with enough memory. The script writes the adapter and the development/held-out metrics to the paths shown in its output.


In [ ]:
RUN_PEFT_TRAINING = False
if RUN_PEFT_TRAINING:
    import subprocess
    subprocess.run([sys.executable, "-m", "evaluation.peft_lora_comparison"], check=True)
else:
    print("PEFT training is disabled. Set RUN_PEFT_TRAINING = True to launch the full GPU training/evaluation run.")


## 8. Generative-model family comparison

The assignment also asks for a comparison of generative-model families. This separate experiment compares a small GRU causal language model, a TF-IDF VAE, and a TF-IDF conditional GAN on BANKING77 intent prediction. It reports train/test accuracy. These compact implementations use different representations and prediction heads; the VAE/GAN scores do not measure chatbot response quality, and this family benchmark is not the prompting/RAG/PEFT comparison above.


Train the three compact family models and evaluate their intent predictions on the official BANKING77 train and test splits. This command writes the measured comparison to `evaluation/results/generative_family_comparison.csv`.


In [ ]:
%run -m evaluation.generative_family_comparison


Load the newly generated family-comparison CSV so the displayed metrics correspond to the code run above.


In [ ]:
family_results = pd.read_csv("evaluation/results/generative_family_comparison.csv")
display(family_results)


## 9. E6 - evaluation and failure analysis (Req 12)

This section compares selected systems row by row on the development subset and surfaces messages that all four systems misclassified. The broader taxonomy and worked cases are documented in `docs/FAILURES.md`, including hallucination cases such as invented labels, unsupported claims, and fabricated tool arguments. Schema validation, code-controlled tool order, and confirmation prompts are the project controls.

In [ ]:
systems = {
    "zero_shot": "evaluation/results/results_zero_shot.csv",
    "few_shot_constraint": "evaluation/results/results_few_shot_constraint.csv",
    "structured": "evaluation/results/results_structured.csv",
    "retrieval": "evaluation/results/results_retrieval.csv",
}
cmp = subset.copy()
for name, path in systems.items():
    cmp[name] = pd.read_csv(path).correct
cmp["n_right"] = cmp[list(systems)].sum(axis=1)
print("accuracy:", {name: round(cmp[name].mean(), 2) for name in systems})
print(f"right in all four: {(cmp.n_right == len(systems)).sum()}, wrong in all four: {(cmp.n_right == 0).sum()}")
retrieval_results = pd.read_csv("evaluation/results/results_retrieval.csv")
display(cmp.loc[cmp.n_right == 0, ["text", "category"]].assign(
    retrieval_output=retrieval_results.loc[cmp.n_right == 0, "output"]
))


## 10. E7 - final tool-assisted assistant (Req 10)

`main.py` combines retrieval, constrained intent classification, schema validation, and code-controlled tools. Action-like requests require confirmation before the simulated tool is called. The code?not the language model?controls the tool sequence, which prevents unsupported tool actions.

Exercise the tool-controlled assistant with an action request, confirmation, question, payment issue, and unknown input.


In [ ]:
from main import handle, TOOLS

print("available tool schemas:", [tool["function"]["name"] for tool in TOOLS])
pending = None
for message in [
    "I lost my card, please block it",
    "yes",
    "Why is it asking me to verify my identity",
    "My top-up for Google Pay isn't working",
    "asdf qwerty zzz",
]:
    print(f"\nCustomer:  {message}")
    reply, pending = handle(message, pending)
    print(f"Assistant: {reply}")


### Final evaluation on the held-out subset

The held-out set contains 100 examples, including every intent, and shares no rows with the development set. Run this final evaluation only after selecting the system using development results; do not tune prompts or retrieval using held-out examples.

Run the final assistant evaluation once on the untouched held-out split.


In [ ]:
%run -m evaluation.final_eval


Report held-out accuracy and inspect a few final-system errors without using them for tuning.


In [ ]:
heldout = pd.read_csv("evaluation/results/results_heldout.csv")
tfidf = pd.read_csv("evaluation/results/results_tfidf_logreg.csv")
print(f"final assistant, held-out accuracy: {heldout.correct.mean():.2f} on {len(heldout)} rows")
print(f"classical reference, development accuracy: {tfidf.correct.mean():.2f} on {len(tfidf)} rows")
heldout.loc[~heldout.correct, ["text", "category", "predicted"]].head(5)


### Optional interactive assistant demo

Set `RUN_INTERACTIVE_DEMO = True` in the next cell to start a text conversation. It is disabled by default so that Run All does not wait for keyboard input.

Optionally start a keyboard-driven chat. It is disabled by default so a Run All operation cannot hang.


In [ ]:
RUN_INTERACTIVE_DEMO = False
if RUN_INTERACTIVE_DEMO:
    from main import handle

    print("Banking assistant ready. Press Enter on an empty line to exit.")
    pending = None
    while True:
        message = input("\nCustomer: ").strip()
        if not message:
            break
        reply, pending = handle(message, pending)
        print("Assistant:", reply)
else:
    print("Interactive chat is disabled. Set RUN_INTERACTIVE_DEMO = True and rerun this cell to start it.")


## 11. Summary

The summary reports measured development and held-out results from the generated CSV files. The causal-LM/VAE/GAN family benchmark is shown separately from the prompting/RAG/PEFT adaptation comparison because they answer different questions.

Collect the generated experiment metrics and show the family and adaptation results separately.


In [ ]:
from src.llm import MODEL
from src.prompts import STYLES
from src.decoding import SETTINGS

read_result = lambda name: pd.read_csv(f"evaluation/results/results_{name}.csv")
accuracy = lambda name: read_result(name).correct.mean()
prompt_scores = {style: accuracy(style) for style in STYLES}
best_prompt = max(prompt_scores, key=prompt_scores.get)
sampling_scores = [accuracy(f"decoding_{setting}") for setting in SETTINGS if setting != "greedy"]
structured = read_result("structured")
family_results = pd.read_csv("evaluation/results/generative_family_comparison.csv")

summary_rows = [
    ("Assistant base model", MODEL),
    ("E1 baseline accuracy (development)", f"{accuracy('baseline'):.2f}"),
    ("Best prompt accuracy (development)", f"{prompt_scores[best_prompt]:.2f} ({best_prompt})"),
    ("Decoding accuracy (development)", f"greedy {accuracy('decoding_greedy'):.2f}; sampling {min(sampling_scores):.2f} to {max(sampling_scores):.2f}"),
    ("Structured output", f"{(~structured.error.str.startswith('not JSON', na=False)).sum()}/{len(structured)} valid JSON; {structured.error.isna().sum()}/{len(structured)} schema-valid"),
    ("TF-IDF + logistic regression", f"{accuracy('tfidf_logreg'):.2f} development accuracy"),
    ("RAG retrieval", f"{accuracy('retrieval'):.2f} development accuracy"),
    ("Final assistant", f"{accuracy('heldout'):.2f} held-out accuracy"),
]
display(pd.DataFrame(summary_rows, columns=["Result", "Measured value"]))
print("\nGenerative-family train/test intent accuracy:")
display(family_results)

peft_file = Path("evaluation/results/results_peft_lora.csv")
if peft_file.exists():
    peft_results = pd.read_csv(peft_file)
    print("\nPEFT results (only if a full run completed):")
    display(peft_results[["method", "split", "examples", "accuracy", "invalid_predictions"]])
else:
    print("\nPEFT: no full-run metric is available yet. Use the optional LoRA training cell on a suitable GPU.")


The final assistant uses retrieval, greedy intent classification, schema validation, and code-controlled tools. The GAN/VAE family benchmark evaluates intent classification, not customer-facing dialogue generation. Full PEFT metrics remain pending until the optional LoRA run completes on suitable hardware.